# Linear Regression Programming

### Dataset
California Housing dataset to predict `MedHouseVal`, the median house value in a California district.

In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

housing = fetch_california_housing(as_frame=True)
df = housing.frame


## Part (a): Inspecting the dataset

In [2]:
display(df.head())

n_observations = df.shape[0]
n_features = df.shape[1] - 1  

print(f"Number of observations: {n_observations}")
print(f"Number of predictor features: {n_features}")

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


Number of observations: 20640
Number of predictor features: 8


**Written response:** Number of observations = 20,640.  Number of predictor features = 8

## Part (b): Create training, validation, and test sets
Spliting the data into 70% training, 15% validation, and 15% test data.

In [3]:
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

RANDOM = 42

# First split off 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM
)

#
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM
)

print(f"Training set:   {X_train.shape[0]} observations")
print(f"Validation set: {X_val.shape[0]} observations")
print(f"Test set:       {X_test.shape[0]} observations")

Training set:   14448 observations
Validation set: 3096 observations
Test set:       3096 observations


**Written response:** I used `RANDOM = 42`. Resulting sizes: training set = 14,448 observations, validation set = 3,096 observations, test set = 3,096 observations.

## Part (c): Baseline model

In [4]:
baseline_prediction = y_train.mean()
val_pred_baseline = np.full(shape=len(y_val), fill_value=baseline_prediction)

baseline_rmse = np.sqrt(mean_squared_error(y_val, val_pred_baseline))

print(f"Baseline prediction (training-set mean): {baseline_prediction:.4f}")
print(f"Baseline validation RMSE: {baseline_rmse:.4f}")

Baseline prediction (training-set mean): 2.0692
Baseline validation RMSE: 1.1414


**Written response:** Baseline validation RMSE = 1.1414 (about $114,140).

## Part (d): Fit two multivariable linear regression models

In [5]:
# Model 1: house-characteristic predictors
features_model1 = ['MedInc', 'AveRooms', 'HouseAge']

# Model 2: income + location predictors
features_model2 = ['MedInc', 'Latitude', 'Longitude', 'Population']

model1 = LinearRegression().fit(X_train[features_model1], y_train)
model2 = LinearRegression().fit(X_train[features_model2], y_train)

print("Model 1 predictors:", features_model1)
print("Model 1 coefficients:", {k: round(float(v), 4) for k, v in zip(features_model1, model1.coef_)})
print("Model 1 intercept:", round(float(model1.intercept_), 4))
print()
print("Model 2 predictors:", features_model2)
print("Model 2 coefficients:", {k: round(float(v), 4) for k, v in zip(features_model2, model2.coef_)})
print("Model 2 intercept:", round(float(model2.intercept_), 4))

Model 1 predictors: ['MedInc', 'AveRooms', 'HouseAge']
Model 1 coefficients: {'MedInc': 0.4419, 'AveRooms': -0.0255, 'HouseAge': 0.0168}
Model 1 intercept: 0.0136

Model 2 predictors: ['MedInc', 'Latitude', 'Longitude', 'Population']
Model 2 coefficients: {'MedInc': 0.3673, 'Latitude': -0.4828, 'Longitude': -0.4952, 'Population': -0.0}
Model 2 intercept: -41.3094


**Written response:** Model 1 predictors = MedInc, AveRooms, HouseAge. Model 2 predictors = MedInc, Latitude, Longitude, Population.

## Part (e): Compare candidate models with validation RMSE

In [6]:
val_pred1 = model1.predict(X_val[features_model1])
val_pred2 = model2.predict(X_val[features_model2])

rmse1 = np.sqrt(mean_squared_error(y_val, val_pred1))
rmse2 = np.sqrt(mean_squared_error(y_val, val_pred2))

print(f"Model 1 validation RMSE: {rmse1:.4f}")
print(f"Model 2 validation RMSE: {rmse2:.4f}")

Model 1 validation RMSE: 0.8168
Model 2 validation RMSE: 0.7546


**Written response:** Model 1 validation RMSE = 0.8168. Model 2 validation RMSE = 0.7546. I selected Model 2 because it had the lower validation RMSE,  its predictions were closer to the true median house values on data the model was not trained on.

## Part (f): Final test evaluation

In [7]:
selected_features = features_model2
selected_model = model2

test_pred = selected_model.predict(X_test[selected_features])
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

print(f"Selected model: Model 2 {selected_features}")
print(f"Test RMSE: {test_rmse:.4f}")

Selected model: Model 2 ['MedInc', 'Latitude', 'Longitude', 'Population']
Test RMSE: 0.7177


**Written response:** Selected model test RMSE = 0.7177. This is close to, and in fact slightly lower than, the validation RMSE of 0.7546, which suggests the model generalizes well and that the validation set gave a reasonable estimate of performance.